# Gutenberg Books to Unity Catalog

This notebook reads text files from Project Gutenberg and writes them to a Unity Catalog table.

## 1. Import Required Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, input_file_name, split
from pyspark.sql.types import StructType, StructField, StringType, LongType
import os
import re

## 2. Initialize Spark Session with Unity Catalog

In [2]:
spark = SparkSession.builder \
    .appName("GutenbergToUnityCatalog") \
    .getOrCreate()

# Set the catalog and schema
spark.sql("USE CATALOG demo-catalog")
spark.sql("USE SCHEMA demo-schema")
print("Spark Session initialized with demo-catalog.demo-schema")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /tmp/.ivy2/cache
The jars for the packages stored in: /tmp/.ivy2/jars
io.delta#delta-spark_2.13 added as a dependency
io.unitycatalog#unitycatalog-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5afb124d-8dd9-41f8-891a-c0190f09925b;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.unitycatalog#unitycatalog-spark_2.13;0.4.0 in central
	found com.fasterxml.jackson.core#jackson-da

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'demo'. SQLSTATE: 42601 (line 1, pos 12)

== SQL ==
USE CATALOG demo-catalog
------------^^^


## 3. Create Schema for Books Data

In [3]:
schema = StructType([
    StructField("book_id", StringType(), True),
    StructField("book_name", StringType(), True),
    StructField("line_number", LongType(), True),
    StructField("line_content", StringType(), True)
])

print("Schema defined for books data")

Schema defined for books data


## 4. Read Gutenberg Books

In [7]:
books_path = "/workspace/data/gutenberg_books"

# Read all text files from the gutenberg books directory
df = spark.read.text(books_path) \
    .select(
        input_file_name().alias("file_path"),
        col("value").alias("line_content")
    )

df = df.withColumn(
    "book_id",
    split(col("file_path"), "/")[-1].substr(1, -5)  # Extract filename without .txt
).withColumn(
    "book_name",
    split(col("file_path"), "/")[-1]
).drop("file_path")

df.show()

26/04/17 14:47:10 ERROR Executor: Exception in task 0.0 in stage 10.0 (TID 33)
org.apache.spark.SparkArrayIndexOutOfBoundsException: [INVALID_ARRAY_INDEX] The index -1 is out of bounds. The array has 7 elements. Use the SQL function `get()` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003
== DataFrame ==
"__getitem__" was called from
line 12 in cell [7]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidArrayIndexError(QueryExecutionErrors.scala:243)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidArrayIndexError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:50)
	at org.apa

ArrayIndexOutOfBoundsException: [INVALID_ARRAY_INDEX] The index -1 is out of bounds. The array has 7 elements. Use the SQL function `get()` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003
== DataFrame ==
"__getitem__" was called from
line 12 in cell [7]


In [ ]:
books_path = "/workspace/data/gutenberg_books"

# Read all text files from the gutenberg books directory
df = spark.read.text(books_path) \
    .select(
        input_file_name().alias("file_path"),
        col("value").alias("line_content")
    )

df.show()

# Extract book ID and name from file path
df = df.withColumn(
    "book_id",
    split(col("file_path"), "/")[-1].substr(1, -5)  # Extract filename without .txt
).withColumn(
    "book_name",
    split(col("file_path"), "/")[-1]
).drop("file_path")

# Add line number
from pyspark.sql.window import Window
window_spec = Window.partitionBy("book_id").orderBy("book_id")
from pyspark.sql.functions import row_number
df = df.withColumn("line_number", row_number().over(window_spec))

# Select columns in correct order
df = df.select("book_id", "book_name", "line_number", "line_content")

print(f"Read {df.count()} lines from gutenberg books")
df.show()

26/04/17 14:46:20 ERROR Executor: Exception in task 5.0 in stage 7.0 (TID 25)
org.apache.spark.SparkArrayIndexOutOfBoundsException: [INVALID_ARRAY_INDEX] The index -1 is out of bounds. The array has 7 elements. Use the SQL function `get()` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003
== DataFrame ==
"__getitem__" was called from
line 13 in cell [5]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidArrayIndexError(QueryExecutionErrors.scala:243)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidArrayIndexError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:50)
	at scala.co

Read 77910 lines from gutenberg books


ArrayIndexOutOfBoundsException: [INVALID_ARRAY_INDEX] The index -1 is out of bounds. The array has 7 elements. Use the SQL function `get()` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003
== DataFrame ==
"__getitem__" was called from
line 13 in cell [5]


## 5. Create Schema (Optional) and Write to Unity Catalog

In [ ]:
# Ensure demo-schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS demo-catalog.demo-schema")
spark.sql("USE SCHEMA demo-schema")
print("Using demo-schema")

## 6. Write DataFrame to Unity Catalog Table

In [ ]:
# Write to Unity Catalog as a managed Delta table
table_name = "gutenberg_books"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(table_name)

print(f"Successfully wrote data to table: {table_name}")

## 7. Verify the Table

In [ ]:
# Read the table back to verify
result_df = spark.sql(f"SELECT * FROM {table_name} LIMIT 20")
result_df.show()

# Show summary statistics
total_lines = spark.sql(f"SELECT COUNT(*) as total_lines FROM {table_name}").collect()[0][0]
unique_books = spark.sql(f"SELECT COUNT(DISTINCT book_id) as unique_books FROM {table_name}").collect()[0][0]

print(f"\nSummary:")
print(f"Total lines: {total_lines}")
print(f"Unique books: {unique_books}")

## 8. Explore the Data

In [ ]:
# Show all books available
spark.sql(f"SELECT DISTINCT book_id, book_name FROM {table_name} ORDER BY book_id").show()

# Show sample content from one book
spark.sql(f"SELECT * FROM {table_name} WHERE book_id = '1342' LIMIT 10").show(truncate=False)